In [ ]:
#Cambiar a entorno virtual y cargar paquetes

import numpy as np
import pandas as pd
from pysyncon import Dataprep, Synth, AugSynth, PenalizedSynth
from pysyncon.utils import PlaceboTest
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
datos = pd.read_csv('../data/dataset_prueba.csv')
datos['date'] = pd.to_datetime(data['date'])
# Describir serie temporal del movimiento en las provincias españolas 
datos.set_index('date', inplace=True, drop=False)

In [ ]:
#Set PANDAS to show all columns in DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [ ]:
print ("# of variables in dataframe:", len(datos.columns))

print  ("# of observations in dataframe:", len(datos))

In [ ]:
np.round(datos.describe(), 2).T[['count','mean', 'std', 'min', 'max']]

In [ ]:
# Filtra los datos solo para la ciudad de 'Zaragoza'
zaragoza_data = data[data['Municipios'] == 'Zaragoza']

# Configura el tamaño del gráfico
plt.figure(figsize=(15, 6))

# # Grafica la serie temporal 'workplaces_percent_change_from_baseline' para Zaragoza
plt.plot(zaragoza_data['date'], zaragoza_data['workplaces_percent_change_from_baseline'], label='Time series for Workplaces', color='blue')

# Grafica la serie temporal 'trend' para Zaragoza
plt.plot(zaragoza_data['date'], zaragoza_data['trend_yearly'], label='Trend', color='red')

# Configura etiquetas y leyenda
plt.xlabel('Fecha')
plt.ylabel('Variación en lugares de trabajo desde la línea de base')
plt.title('Comparación entre valores reales y su tendencia en Zaragoza')
plt.legend()

# Muestra el gráfico
plt.show()


In [ ]:
#Comenzamos análisis exxploratorio ))
print(data[data.date.duplicated()]['date'].count())
print('Fecha inicio del histórico: ', data.date.min())
print('Fecha fin del histórico: ', data.date.max())
print('Número total de días: ', data.date.nunique())

In [ ]:
# data = data[data['Población residente (Personas)'] > 200]

In [ ]:
#Prueba pysyncon
municipios_control = valores_sin_zaragoza = [municipio for municipio in data['Municipios'].unique() if municipio != "Zaragoza"]

dataprep = Dataprep(
    foo=data,
    predictors=[
         'Edad mediana de la población (años)',
         'Número medio de hijos por mujer (Número)',
         'Población residente (Personas)',
         'Superficie total (Km2)',
         'Proporción de empleo en industria (NACE Rev.2 B-E) (Porcentaje)',
         'Proporción de empleo en servicios (NACE Rev.2 G-U) (Porcentaje)',
         'Proporción de extranjeros sobre la población total (Porcentaje)',
         'Proporción de población entre 25-64 años con máximo nivel educación ISCED 0, 1 ó 2 (Porcentaje)',
         'Proporción de población entre 25-64 años con máximo nivel de educación ISCED 3 ó 4 (Porcentaje)',
         'Proporción de población de 15-64 años (Porcentaje)',
         'Renta neta media anual por habitante (Euros)',
         'Tasa de desempleo (Porcentaje)',
    ],
    predictors_op="mean",
    time_predictors_prior=range(30, 75),
    special_predictors=[
        # ('Valor_Stringency', range(30,75), 'mean'),
        # ('Valor_GovernmentResponseIndex',  range(30,75), 'mean'), 
        # ('Valor_EconomicSupportIndex',  range(30,75), 'mean'),
        # ('Valor_Containment',  range(30,75), 'mean'),
    ],
    dependent="trend_yearly",
    unit_variable="Municipios",
    time_variable="Periodo",
    treatment_identifier='Zaragoza',
    controls_identifier=municipios_control,
    time_optimize_ssr=range(30, 127),
)

In [ ]:
pen = PenalizedSynth()
pen.fit(dataprep, lambda_=0.01)

In [ ]:

print(pen.weights())

In [ ]:
pen.path_plot(time_period=range(1, 202), treatment_time=127)

In [ ]:
pen.summary()

In [ ]:
synth = Synth()
synth.fit(dataprep=dataprep, optim_method="Nelder-Mead", optim_initial="ols")

In [ ]:
synth.weights()
synth.path_plot(time_period=range(1, 202), treatment_time=127)
synth.gaps_plot(time_period=range(1, 202), treatment_time=127)
synth.summary()

synth.weights()

In [ ]:
augsynth = AugSynth()
augsynth.fit(dataprep=dataprep)

In [ ]:
print(augsynth.weights())

In [ ]:
augsynth.path_plot(time_period=range(1, 202), treatment_time=127)

In [ ]:
augsynth.gaps_plot(time_period=range(1, 365), treatment_time=202)

In [ ]:
augsynth.cv_result.plot()

In [ ]:
augsynth.summary()